# Logit Models

Converted from `Code/r_01_logits.R` — R-kernel Jupyter port. Outputs tables to `Output/Tables/` and figures to `Output/Images/Graphs/`. Run the setup cell first, then sections in order.

In [1]:
pacman::p_load(
  sf, tidyverse, stargazer, dplyr,
  raster, spdep, sp, ggplot2, robust,
  lmtest, sandwich, car, jsonlite, conleyreg
)

PROJECT_ROOT <- tryCatch(
  normalizePath(file.path(dirname(rstudioapi::getActiveDocumentContext()$path), "..")),
  error = function(e) {
    cwd <- normalizePath(getwd())
    if (basename(cwd) == "Code") dirname(cwd) else cwd
  }
)
setwd(PROJECT_ROOT)
# Load pretty dictionary for labels
pretty_dict <- fromJSON("Code/pretty_dict.json")

pdf <- read_sf(dsn = "Data/Processed/northParishFlows.shp")

## Section 1: Main Parish-Level Logit Models

Monastic variables per arable km²

In [2]:
# Standardize and center continuous variables.
# Binary dummies (smHouse, bigHouse, mg_fsnub, mg_court, friary) are NOT standardized.
for (v in c(
  # Total monastic land (3 normalizations)
  "llandOwned", "llo_sk", "llo_arak",
  # Small/large split land (3 normalizations)
  "lsmLand", "lbigLand",
  "lsm_sk",  "lbg_sk",
  "lsm_arak","lbg_arak",
  # Off-site/on-site split land (3 normalizations)
  "lotherLand", "lownLand",
  "loth_sk",    "lown_sk",
  "loth_arak",  "lown_arak",
  # Tithes, alms, net income (3 normalizations each)
  "ltitheOutT", "lti_sk", "lti_arak",
  "lalmsInTot", "lal_sk", "lal_arak",
                "lni_sk", "lni_arak",
  # Controls (continuous)
  "lLStax_pc", "wet_1535", "wet_1536", "lpopC",
  "area", "mean_slope", "distScot"
)) {
  pdf[[v]] <- scale(pdf[[v]], center = TRUE, scale = TRUE)[, 1]
}

# --- Controls shared across all specifications ----------------------------
# mg_fsnub / mg_court: 20 km binary proximity dummies — not standardized.
controls <- c("mg_fsnub", "mg_court",
              "lLStax_pc", "wet_1535", "wet_1536", "lpopC",
              "uplands", "lowlands", "area", "mean_slope", "distScot")
hide_vars <- c("Constant", "uplands", "lowlands", "area", "mean_slope")
full_controls_formula <- paste(controls, collapse = " + ")

# --- Nine monastic-variable specifications --------------------------------
monastic_specs <- list(
  total_raw = list(
    monastic_vars = c("llandOwned", "ltitheOutT", "lalmsInTot",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("llandOwned", "smHouse", "bigHouse"),
      c("ltitheOutT", "lalmsInTot", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_total_raw"
  ),
  total_sk = list(
    monastic_vars = c("llo_sk", "lti_sk", "lal_sk",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("llo_sk", "smHouse", "bigHouse"),
      c("lti_sk", "lal_sk", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_total_sk"
  ),
  total_arak = list(
    monastic_vars = c("llo_arak", "lti_arak", "lal_arak", "lni_arak",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("llo_arak", "smHouse", "bigHouse"),
      c("lti_arak", "lal_arak", "lni_arak", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_total_arak"
  ),
  split_raw = list(
    monastic_vars = c("lsmLand", "lbigLand", "ltitheOutT", "lalmsInTot",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("lsmLand", "lbigLand", "smHouse", "bigHouse"),
      c("ltitheOutT", "lalmsInTot", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_split_raw"
  ),
  split_sk = list(
    monastic_vars = c("lsm_sk", "lbg_sk", "lti_sk", "lal_sk",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("lsm_sk", "lbg_sk", "smHouse", "bigHouse"),
      c("lti_sk", "lal_sk", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_split_sk"
  ),
  split_arak = list(
    monastic_vars = c("lsm_arak", "lbg_arak", "lti_arak", "lal_arak", "lni_arak",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("lsm_arak", "lbg_arak", "smHouse", "bigHouse"),
      c("lti_arak", "lal_arak", "lni_arak", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_split_arak"
  ),
  ownOther_raw = list(
    monastic_vars = c("lotherLand", "lownLand", "ltitheOutT", "lalmsInTot",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("lotherLand", "lownLand", "smHouse", "bigHouse"),
      c("ltitheOutT", "lalmsInTot", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_ownOther_raw"
  ),
  ownOther_sk = list(
    monastic_vars = c("loth_sk", "lown_sk", "lti_sk", "lal_sk",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("loth_sk", "lown_sk", "smHouse", "bigHouse"),
      c("lti_sk", "lal_sk", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_ownOther_sk"
  ),
  ownOther_arak = list(
    monastic_vars = c("loth_arak", "lown_arak", "lti_arak", "lal_arak", "lni_arak",
                      "smHouse", "bigHouse", "friary"),
    var_list_list = list(
      c("loth_arak", "lown_arak", "smHouse", "bigHouse"),
      c("lti_arak", "lal_arak", "lni_arak", "friary"),
      c("lLStax_pc", "lpopC", "wet_1535", "wet_1536"),
      c("uplands", "lowlands", "area", "mean_slope")
    ),
    suffix = "_ownOther_arak"
  )
)

# --- Helper: coefficient plot for a full GLM model -----------------------
make_logit_plot <- function(model, plot_vars, pretty_dict,
                            x_label = "Coefficient (Log Odds / Log Count)") {
  cs        <- summary(model)$coefficients
  coef_rows <- lapply(plot_vars, function(v) {
    if (!v %in% rownames(cs)) return(NULL)
    est <- cs[v, "Estimate"];  se <- cs[v, "Std. Error"]
    data.frame(variable  = v,
               coefficient = est, se = se,
               ci_lower  = est - 1.645 * se,
               ci_upper  = est + 1.645 * se,
               p_value   = cs[v, ncol(cs)])
  })
  coef_df <- bind_rows(coef_rows)
  if (nrow(coef_df) == 0) return(NULL)
  labels_vec             <- unlist(pretty_dict)
  coef_df$variable_label <- unname(labels_vec[coef_df$variable])
  coef_df$significant    <- coef_df$p_value < 0.10
  coef_df$order          <- match(coef_df$variable, plot_vars)
  lvl_order              <- order(-coef_df$order)
  coef_df$variable_label <- factor(coef_df$variable_label,
                                    levels = coef_df$variable_label[lvl_order])
  ggplot(coef_df, aes(x = coefficient, y = variable_label)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    geom_errorbar(aes(xmin = ci_lower, xmax = ci_upper),
                  width = 0.2, color = "gray30", orientation = "y") +
    geom_point(aes(color = significant), size = 3) +
    scale_color_manual(values = c("FALSE" = "gray60", "TRUE" = "#0072B2"),
                       labels = c("FALSE" = "Not Significant", "TRUE" = "p < 0.10")) +
    labs(x = x_label, y = "", color = "Significance") +
    theme_minimal() +
    theme(axis.text.x    = element_text(size = 14),
          axis.text.y    = element_text(size = 14),
          axis.title.x   = element_text(size = 14),
          legend.text    = element_text(size = 13),
          legend.title   = element_text(size = 13),
          legend.position = "bottom")
}

# --- Helper: Conley (spatial HAC) SEs at 100 km for a GLM model ----------
# pdf is the sf object read at the top; conleyreg uses its geometry for distances.
conley_glm_se <- function(glm_mod, family_lab, cutoff_km = 100) {
  fm <- formula(glm_mod)
  cr <- tryCatch(
    conleyreg(
      formula     = fm,
      data        = pdf,
      dist_cutoff = cutoff_km,
      model       = family_lab,
      kernel      = "bartlett",
      verbose     = FALSE
    ),
    error = function(e) {
      cat("conleyreg failed:", conditionMessage(e), "\n"); NULL
    }
  )
  if (is.null(cr)) return(NULL)
  out <- setNames(rep(NA_real_, length(coef(glm_mod))), names(coef(glm_mod)))
  out[rownames(cr)] <- cr[, "Std. Error"]
  out
}

# --- Helper: run the full monastic analysis for one spec -----------------
run_monastic_spec <- function(spec_name, spec) {
  mon_vars      <- spec$monastic_vars
  var_list_list <- spec$var_list_list
  sfx           <- spec$suffix

  mon_cov_order  <- c(mon_vars, "mg_fsnub", "mg_court",
                      "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  mon_cov_labels <- unlist(pretty_dict[mon_cov_order])

  # (a) Single-variable logit tables — one column per monastic var ----------
  muster_list <- list(); primary_list <- list(); seat_list <- list()
  for (var in mon_vars) {
    muster_list[[var]]  <- glm(paste("muster ~",  var, "+", full_controls_formula),
                               data = pdf, family = binomial(link = "logit"))
    primary_list[[var]] <- glm(paste("primary ~", var, "+", full_controls_formula),
                               data = pdf, family = binomial(link = "logit"))
    seat_list[[var]]    <- glm(paste("seats ~",   var, "+", full_controls_formula),
                               data = pdf, family = "poisson")
  }
  muster_ses  <- lapply(muster_list,  function(m) conley_glm_se(m, "logit"))
  primary_ses <- lapply(primary_list, function(m) conley_glm_se(m, "logit"))
  seat_ses    <- lapply(seat_list,    function(m) conley_glm_se(m, "poisson"))

  stargazer(muster_list, type = "latex",
    title   = paste0("Muster — Monastic Variables [", spec_name, "]"),
    label   = paste0("tab:muster_monastic", sfx),
    se      = muster_ses,
    omit    = hide_vars, covariate.labels = mon_cov_labels,
    add.lines = list(c("Controls", rep("Y", length(mon_vars))),
                     c("Conley SEs (100 km)", rep("Y", length(mon_vars)))),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/muster_monastic", sfx, ".tex"))
  stargazer(primary_list, type = "latex",
    title   = paste0("Primary — Monastic Variables [", spec_name, "]"),
    label   = paste0("tab:primary_monastic", sfx),
    se      = primary_ses,
    omit    = hide_vars, covariate.labels = mon_cov_labels,
    add.lines = list(c("Controls", rep("Y", length(mon_vars))),
                     c("Conley SEs (100 km)", rep("Y", length(mon_vars)))),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/primary_monastic", sfx, ".tex"))
  stargazer(seat_list, type = "latex",
    title   = paste0("Seats — Monastic Variables [", spec_name, "]"),
    label   = paste0("tab:seat_monastic", sfx),
    se      = seat_ses,
    omit    = hide_vars, covariate.labels = mon_cov_labels,
    add.lines = list(c("Controls", rep("Y", length(mon_vars))),
                     c("Conley SEs (100 km)", rep("Y", length(mon_vars)))),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/seat_monastic", sfx, ".tex"))

  # (b) Progressive model building -----------------------------------------
  fit_prog <- function(dep, family) {
    out <- list(); vl <- c(); i <- 1
    for (vs in var_list_list) {
      vl <- c(vl, vs)
      out[[i]] <- glm(paste(dep, "~", paste(vl, collapse = " + ")),
                      data = pdf, family = family)
      i <- i + 1
    }
    out
  }
  muster_prog  <- fit_prog("muster",  binomial(link = "logit"))
  primary_prog <- fit_prog("primary", binomial(link = "logit"))
  seat_prog    <- fit_prog("seats",   "poisson")

  muster_prog_ses  <- lapply(muster_prog,  function(m) conley_glm_se(m, "logit"))
  primary_prog_ses <- lapply(primary_prog, function(m) conley_glm_se(m, "logit"))
  seat_prog_ses    <- lapply(seat_prog,    function(m) conley_glm_se(m, "poisson"))

  prog_order      <- unlist(var_list_list[1:3])
  cov_labels_prog <- unlist(pretty_dict[prog_order])
  stargazer(muster_prog, type = "latex",
    title = paste0("Muster — Progressive Model [", spec_name, "]"),
    label = paste0("tab:muster_all", sfx),
    se    = muster_prog_ses,
    omit  = hide_vars, covariate.labels = cov_labels_prog,
    add.lines = list(c("Controls", "N", "N", "N", "Y"),
                     c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/muster_all", sfx, ".tex"))
  stargazer(primary_prog, type = "latex",
    title = paste0("Primary — Progressive Model [", spec_name, "]"),
    label = paste0("tab:primary_all", sfx),
    se    = primary_prog_ses,
    omit  = hide_vars, covariate.labels = cov_labels_prog,
    add.lines = list(c("Controls", "N", "N", "N", "Y"),
                     c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/primary_all", sfx, ".tex"))
  stargazer(seat_prog, type = "latex",
    title = paste0("Seats — Progressive Model [", spec_name, "]"),
    label = paste0("tab:seat_all", sfx),
    se    = seat_prog_ses,
    omit  = hide_vars, covariate.labels = cov_labels_prog,
    add.lines = list(c("Controls", "N", "N", "N", "Y"),
                     c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/seat_all", sfx, ".tex"))

  # (c) Full joint model — muster / primary / seats in columns -------------
  all_mon_formula <- paste(mon_vars, collapse = " + ")
  full_muster  <- glm(paste("muster ~",  all_mon_formula, "+", full_controls_formula),
                      data = pdf, family = binomial(link = "logit"))
  full_primary <- glm(paste("primary ~", all_mon_formula, "+", full_controls_formula),
                      data = pdf, family = binomial(link = "logit"))
  full_seat    <- glm(paste("seats ~",   all_mon_formula, "+", full_controls_formula),
                      data = pdf, family = "poisson")

  full_muster_se  <- conley_glm_se(full_muster,  "logit")
  full_primary_se <- conley_glm_se(full_primary, "logit")
  full_seat_se    <- conley_glm_se(full_seat,    "poisson")

  stargazer(full_muster, full_primary, full_seat, type = "latex",
    title   = paste0("Full Monastic Model [", spec_name, "]"),
    label   = paste0("tab:full_monastic", sfx),
    se      = list(full_muster_se, full_primary_se, full_seat_se),
    omit    = hide_vars, covariate.labels = mon_cov_labels,
    column.labels = c("Muster", "Primary", "Seats"),
    add.lines = list(c("Conley SEs (100 km)", "Y", "Y", "Y")),
    align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"),
    table.placement = "H",
    out = paste0("Output/Tables/full_monastic", sfx, ".tex"))

  # (d) Coefficient plots for the full joint model
  plot_vars <- c(mon_vars, "mg_fsnub", "mg_court",
                 "lLStax_pc", "wet_1535", "wet_1536", "lpopC")
  for (nm in c("muster", "primary", "seats")) {
    mod <- switch(nm, muster = full_muster, primary = full_primary, seats = full_seat)
    p   <- make_logit_plot(mod, plot_vars, pretty_dict)
    if (!is.null(p)) {
      ggsave(paste0("Output/Images/Graphs/logit_", nm, sfx, ".png"),
             plot = p, width = 10, height = 6, dpi = 300)
    }
  }

  list(muster = full_muster, primary = full_primary, seat = full_seat)
}

# --- Run all nine specs ---------------------------------------------------
full_models_by_spec <- list()
for (spec_name in names(monastic_specs)) {
  full_models_by_spec[[spec_name]] <- run_monastic_spec(
    spec_name, monastic_specs[[spec_name]]
  )
}

# Back-compat aliases: split_arak is the primary per-arable-km² specification
monastic_vars <- monastic_specs$split_arak$monastic_vars
full_muster   <- full_models_by_spec$split_arak$muster
full_primary  <- full_models_by_spec$split_arak$primary
full_seat     <- full_models_by_spec$split_arak$seat

# --- DAG Regressions (minimal adjustment set) ----------------------------
dag_vars_labels <- c("llo_arak", "lpopC", "lLStax_pc")
dag_cov_labels  <- unlist(pretty_dict[dag_vars_labels])

dag_muster  <- glm(muster  ~ llo_arak + lpopC + lLStax_pc,
                   data = pdf, family = binomial(link = "logit"))
dag_primary <- glm(primary ~ llo_arak + lpopC + lLStax_pc,
                   data = pdf, family = binomial(link = "logit"))
dag_seat    <- glm(seats   ~ llo_arak + lpopC + lLStax_pc,
                   data = pdf, family = "poisson")

dag_muster_se  <- conley_glm_se(dag_muster,  "logit")
dag_primary_se <- conley_glm_se(dag_primary, "logit")
dag_seat_se    <- conley_glm_se(dag_seat,    "poisson")

stargazer(dag_muster, dag_primary, dag_seat,
  type = "latex", title = "DAG Results", label = "tab:dag",
  se   = list(dag_muster_se, dag_primary_se, dag_seat_se),
  align = TRUE, column.sep.width = ".5pt",
  covariate.labels = dag_cov_labels,
  column.labels    = c("Muster", "Primary", "Seats"),
  add.lines        = list(c("Conley SEs (100 km)", "Y", "Y", "Y")),
  omit.stat        = c("aic"), table.placement = "H",
  out = "Output/Tables/dag.tex"
)

Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


...


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:09:57
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Full Monastic Model [ownOther_arak]} 
  \label{tab:full_monastic_ownOther_arak} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{muster \textasciitilde} & \multicolumn{1}{c}{primary \textasciitilde} & \multicolumn{1}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{Poisson}} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Off-site 

Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:09:59
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{DAG Results} 
  \label{tab:dag} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{muster} & \multicolumn{1}{c}{primary} & \multicolumn{1}{c}{seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{Poisson}} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln(Land Owned / Arable km²) & 0.263^{*} & 0.433^{**} & 0.102 \\ 
  & (0.144) & (0.200) & (0.092) \\ 
  & & &

## VIF Analysis for Multicollinearity Detection

In [3]:
cat("\n========== VIF ANALYSIS ==========\n\n")

vif_table <- function(model, model_name) {
  cat(paste0("Model: ", model_name, "\n"))
  cat("-----------------------------------\n")
  vif_values <- vif(model)
  vif_df <- data.frame(Variable = names(vif_values), VIF = round(vif_values, 3))
  vif_df <- vif_df[order(-vif_df$VIF), ]
  print(vif_df)
  cat("\n")
  return(vif_df)
}

cat("=== FULL MONASTIC MODELS (all 9 specs) ===\n\n")
for (spec_name in names(full_models_by_spec)) {
  mods <- full_models_by_spec[[spec_name]]
  vif_table(mods$muster,  paste("Muster -",  spec_name))
  vif_table(mods$primary, paste("Primary -", spec_name))
  vif_table(mods$seat,    paste("Seats -",   spec_name))
}

cat("\n=== DAG MODELS ===\n\n")
vif_table(dag_muster,  "Muster - DAG Model")
vif_table(dag_primary, "Primary - DAG Model")
vif_table(dag_seat,    "Seats - DAG Model")

cat("\n========== VIF SUMMARY & RECOMMENDATIONS ==========\n")
cat("VIF Interpretation:\n")
cat("  VIF < 5:    Low multicollinearity (generally acceptable)\n")
cat("  VIF 5-10:   Moderate multicollinearity (use with caution)\n")
cat("  VIF > 10:   High multicollinearity (problematic)\n\n")

all_vif_results <- list()
for (spec_name in names(full_models_by_spec)) {
  all_vif_results[[paste0("full_muster_",  spec_name)]] <- vif(full_models_by_spec[[spec_name]]$muster)
  all_vif_results[[paste0("full_primary_", spec_name)]] <- vif(full_models_by_spec[[spec_name]]$primary)
  all_vif_results[[paste0("full_seat_",    spec_name)]] <- vif(full_models_by_spec[[spec_name]]$seat)
}

high_vif <- sapply(all_vif_results, function(x) any(x > 10))
if (any(high_vif)) {
  cat("WARNING: High VIF (>10) detected in:\n")
  for (name in names(high_vif)[high_vif]) {
    high_vars <- names(all_vif_results[[name]][all_vif_results[[name]] > 10])
    cat(paste("  -", name, ":", paste(high_vars, collapse = ", "), "\n"))
  }
} else {
  cat("No high VIF (>10) detected. Multicollinearity appears manageable.\n")
}
cat("\n")



========== VIF ANALYSIS ==========



=== FULL MONASTIC MODELS (all 9 specs) ===



Model: Muster - total_raw
-----------------------------------
             Variable   VIF
wet_1536     wet_1536 2.450
mean_slope mean_slope 2.199
wet_1535     wet_1535 2.067
area             area 1.900
distScot     distScot 1.710
bigHouse     bigHouse 1.688
mg_court     mg_court 1.688
mg_fsnub     mg_fsnub 1.668
uplands       uplands 1.540
llandOwned llandOwned 1.531
lowlands     lowlands 1.426
lLStax_pc   lLStax_pc 1.397
lalmsInTot lalmsInTot 1.384
ltitheOutT ltitheOutT 1.381
lpopC           lpopC 1.336
smHouse       smHouse 1.287
friary         friary 1.091

Model: Primary - total_raw
-----------------------------------
             Variable   VIF
mean_slope mean_slope 2.442
wet_1536     wet_1536 2.356
area             area 1.925
wet_1535     wet_1535 1.907
bigHouse     bigHouse 1.866
mg_court     mg_court 1.790
llandOwned llandOwned 1.789
distScot     distScot 1.732
mg_fsnub     mg_fsnub 1.726
uplands       uplands 1.693
lowlands     lowlands 1.608
lalmsInTot lalmsInTot 1.541
smHous


=== DAG MODELS ===



Model: Muster - DAG Model
-----------------------------------
           Variable   VIF
lLStax_pc lLStax_pc 1.049
llo_arak   llo_arak 1.037
lpopC         lpopC 1.024



...

VIF Interpretation:


  VIF < 5:    Low multicollinearity (generally acceptable)


  VIF 5-10:   Moderate multicollinearity (use with caution)


  VIF > 10:   High multicollinearity (problematic)



No high VIF (>10) detected. Multicollinearity appears manageable.


## Section 2: Distance-Weighted Interaction Models (Robustness)

Spatially-weighted monastic variables (raw, per-capita, per-sq-km)

In [4]:
# Standardize distance-weighted variables
dw_vars_all <- c(
  "llo_dw",   "lsl_dw",   "lbl_dw",   "lti_dw",
  "llo_dwpc", "lsl_dwpc", "lbl_dwpc", "lti_dwpc",
  "llo_dwsk", "lsl_dwsk", "lbl_dwsk", "lti_dwsk"
)
for (v in dw_vars_all) {
  pdf[[v]] <- scale(pdf[[v]], center = TRUE, scale = TRUE)[, 1]
}
pdf$Y_COORD <- scale(pdf$Y_COORD, center = TRUE, scale = TRUE)[, 1]

controls_dw <- c(
  "mg_fsnub", "mg_court",
  "lLStax_pc", "wet_1535", "wet_1536", "lpopC",
  "Y_COORD", "uplands", "lowlands", "area", "mean_slope", "distScot"
)
hide_vars_dw <- c("Constant", "Y_COORD", "uplands", "lowlands", "area", "mean_slope")

run_models_dw <- function(monastic_vars, pdf, controls) {
  muster_list  <- list()
  primary_list <- list()
  seat_list    <- list()
  for (var in monastic_vars) {
    f_base <- paste(controls, collapse = " + ")
    muster_list[[var]]  <- glm(paste("muster ~",  var, "+", f_base), data = pdf,
                                family = binomial(link = "logit"))
    primary_list[[var]] <- glm(paste("primary ~", var, "+", f_base), data = pdf,
                                family = binomial(link = "logit"))
    seat_list[[var]]    <- glm(paste("seats ~",   var, "+", f_base), data = pdf,
                                family = "poisson")
  }
  list(muster = muster_list, primary = primary_list, seat = seat_list)
}

# Helper: Conley SEs for a list of GLM models
conley_list_se <- function(model_list, family_lab, cutoff_km = 100) {
  lapply(model_list, function(m) conley_glm_se(m, family_lab, cutoff_km))
}

# --- Set 1: Raw distance-weighted ---
dw_raw <- c("llo_dw", "lsl_dw", "lbl_dw", "lti_dw")
res_raw <- run_models_dw(dw_raw, pdf, controls_dw)

raw_labels <- unlist(pretty_dict[c(dw_raw, "mg_fsnub", "mg_court",
                                   "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")])

stargazer(res_raw$muster,
  type = "latex", title = "Muster: Raw Distance-Weighted Monastic Variables",
  label = "tab:muster_dw_raw",
  se = conley_list_se(res_raw$muster, "logit"),
  omit = hide_vars_dw, covariate.labels = raw_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/muster_dw_raw.tex"
)
stargazer(res_raw$primary,
  type = "latex", title = "Primary: Raw Distance-Weighted Monastic Variables",
  label = "tab:primary_dw_raw",
  se = conley_list_se(res_raw$primary, "logit"),
  omit = hide_vars_dw, covariate.labels = raw_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/primary_dw_raw.tex"
)
stargazer(res_raw$seat,
  type = "latex", title = "Seats: Raw Distance-Weighted Monastic Variables",
  label = "tab:seat_dw_raw",
  se = conley_list_se(res_raw$seat, "poisson"),
  omit = hide_vars_dw, covariate.labels = raw_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/seat_dw_raw.tex"
)

# --- Set 2: Per capita distance-weighted ---
dw_pc <- c("llo_dwpc", "lsl_dwpc", "lbl_dwpc", "lti_dwpc")
res_pc <- run_models_dw(dw_pc, pdf, controls_dw)

pc_labels <- unlist(pretty_dict[c(dw_pc, "mg_fsnub", "mg_court",
                                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")])

stargazer(res_pc$muster,
  type = "latex", title = "Muster: Per Capita Distance-Weighted Monastic Variables",
  label = "tab:muster_dw_pc",
  se = conley_list_se(res_pc$muster, "logit"),
  omit = hide_vars_dw, covariate.labels = pc_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/muster_dw_pc.tex"
)
stargazer(res_pc$primary,
  type = "latex", title = "Primary: Per Capita Distance-Weighted Monastic Variables",
  label = "tab:primary_dw_pc",
  se = conley_list_se(res_pc$primary, "logit"),
  omit = hide_vars_dw, covariate.labels = pc_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/primary_dw_pc.tex"
)
stargazer(res_pc$seat,
  type = "latex", title = "Seats: Per Capita Distance-Weighted Monastic Variables",
  label = "tab:seat_dw_pc",
  se = conley_list_se(res_pc$seat, "poisson"),
  omit = hide_vars_dw, covariate.labels = pc_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/seat_dw_pc.tex"
)

# --- Set 3: Per sq km distance-weighted ---
dw_sk <- c("llo_dwsk", "lsl_dwsk", "lbl_dwsk", "lti_dwsk")
res_sk <- run_models_dw(dw_sk, pdf, controls_dw)

sk_labels <- unlist(pretty_dict[c(dw_sk, "mg_fsnub", "mg_court",
                                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")])

stargazer(res_sk$muster,
  type = "latex", title = "Muster: Per Sq Km Distance-Weighted Monastic Variables",
  label = "tab:muster_dw_sk",
  se = conley_list_se(res_sk$muster, "logit"),
  omit = hide_vars_dw, covariate.labels = sk_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/muster_dw_sk.tex"
)
stargazer(res_sk$primary,
  type = "latex", title = "Primary: Per Sq Km Distance-Weighted Monastic Variables",
  label = "tab:primary_dw_sk",
  se = conley_list_se(res_sk$primary, "logit"),
  omit = hide_vars_dw, covariate.labels = sk_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/primary_dw_sk.tex"
)
stargazer(res_sk$seat,
  type = "latex", title = "Seats: Per Sq Km Distance-Weighted Monastic Variables",
  label = "tab:seat_dw_sk",
  se = conley_list_se(res_sk$seat, "poisson"),
  omit = hide_vars_dw, covariate.labels = sk_labels,
  add.lines = list(c("Geographic Controls", "Y", "Y", "Y", "Y"),
                   c("Conley SEs (100 km)", "Y", "Y", "Y", "Y")),
  align = TRUE, column.sep.width = ".5pt", omit.stat = c("aic"), table.placement = "H",
  out = "Output/Tables/seat_dw_sk.tex"
)


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:01
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Muster: Raw Distance-Weighted Monastic Variables} 
  \label{tab:muster_dw_raw} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{muster \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 ln(Distance-Weighted Land Owned) & 0.213^{**} &  &  &  \\ 
  & (0.083) &  &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Small House Land) &  & -0.018 &  &  \\ 
  &  & (0.109) &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Big House Land) &  &  & 0.208^{**} &  \\ 
  &  &  & (0.082) &  \\ 
  &


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:02
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Primary: Raw Distance-Weighted Monastic Variables} 
  \label{tab:primary_dw_raw} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{primary \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 ln(Distance-Weighted Land Owned) & 0.227^{***} &  &  &  \\ 
  & (0.086) &  &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Small House Land) &  & -0.963^{*} &  &  \\ 
  &  & (0.533) &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Big House Land) &  &  & 0.239^{***} &  \\ 
  &  &  & (0.086) &


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:03
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Seats: Raw Distance-Weighted Monastic Variables} 
  \label{tab:seat_dw_raw} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 ln(Distance-Weighted Land Owned) & 0.085^{**} &  &  &  \\ 
  & (0.038) &  &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Small House Land) &  & 0.175^{***} &  &  \\ 
  &  & (0.033) &  &  \\ 
  & & & & \\ 
 ln(Distance-Weighted Big House Land) &  &  & 0.050 &  \\ 
  &  &  & (0.034) &  \\ 
  & & &

Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"


Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"


...


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:06
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Seats: Per Capita Distance-Weighted Monastic Variables} 
  \label{tab:seat_dw_pc} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 Snubbed Family Proximity (20 km) & 0.00002 &  &  &  \\ 
  & (0.186) &  &  &  \\ 
  & & & & \\ 
 Court Officer Proximity (20 km) &  & 0.240 &  &  \\ 
  &  & (0.162) &  &  \\ 
  & & & & \\ 
 ln(Lay Subsidy per Capita) &  &  & -0.174 &  \\ 
  &  &  & (0.220) &  \\ 
  & & & & \\ 
 Wet 1535 We

Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:07
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Muster: Per Sq Km Distance-Weighted Monastic Variables} 
  \label{tab:muster_dw_sk} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{muster \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 Snubbed Family Proximity (20 km) & 2.135 &  &  &  \\ 
  & (1.807) &  &  &  \\ 
  & & & & \\ 
 Court Officer Proximity (20 km) &  & 0.829 &  &  \\ 
  &  & (0.701) &  &  \\ 
  & & & & \\ 
 ln(Lay Subsidy per Capita) &  &  & 1.664 &  \\ 
  &  &  & (1.476) &  \\ 
  & & & & \\ 
 Wet 1535 We


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:08
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Primary: Per Sq Km Distance-Weighted Monastic Variables} 
  \label{tab:primary_dw_sk} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{primary \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 Snubbed Family Proximity (20 km) & 1.736 &  &  &  \\ 
  & (2.264) &  &  &  \\ 
  & & & & \\ 
 Court Officer Proximity (20 km) &  & -37.233 &  &  \\ 
  &  & (28.737) &  &  \\ 
  & & & & \\ 
 ln(Lay Subsidy per Capita) &  &  & 2.234 &  \\ 
  &  &  & (1.516) &  \\ 
  & & & & \\ 
 Wet 1


% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:10
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Seats: Per Sq Km Distance-Weighted Monastic Variables} 
  \label{tab:seat_dw_sk} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
\cline{2-5} 
\\[-1.8ex] & \multicolumn{4}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)}\\ 
\hline \\[-1.8ex] 
 Snubbed Family Proximity (20 km) & -0.096 &  &  &  \\ 
  & (1.131) &  &  &  \\ 
  & & & & \\ 
 Court Officer Proximity (20 km) &  & 0.942 &  &  \\ 
  &  & (0.738) &  &  \\ 
  & & & & \\ 
 ln(Lay Subsidy per Capita) &  &  & -1.294 &  \\ 
  &  &  & (1.117) &  \\ 
  & & & & \\ 
 Wet 1535 Weat

## Section 3: Conley (Spatial HAC) Standard Errors — Robustness

Re-estimate the full monastic models (`total` spec) and the joint elite model
using Conley spatial HAC standard errors. Inter-parish distances are computed
on centroids of `northParishFlows.shp` (EPSG:27700, BNG meters; converted to km).
We report SEs at three cutoff distances — **50, 100, 150 km** — using a
Bartlett kernel. The original GLM SEs are reported alongside for comparison.

In [5]:
# Conley (spatial HAC) standard errors via the `conleyreg` package.
# For each of the 9 monastic specs, re-estimate the full joint model using
# Conley SEs at multiple distance cutoffs and write one .tex per spec/cutoff.

pacman::p_load(conleyreg)

# Parish centroids in BNG (EPSG:27700) — conleyreg accepts the sf object directly.
pdf_sf <- pdf  # already an sf object

# Cutoffs in km
conley_cutoffs <- c(50, 100, 150)

# --- Helper: Conley SEs aligned to glm coefficient names -----------------
conley_se <- function(glm_mod, cutoff_km, family_lab) {
  fm <- formula(glm_mod)
  cr <- conleyreg(
    formula     = fm,
    data        = pdf_sf,
    dist_cutoff = cutoff_km,
    model       = family_lab,   # "logit" or "poisson"
    kernel      = "bartlett",
    verbose     = FALSE
  )
  cr_se <- cr[, "Std. Error"]
  out   <- setNames(rep(NA_real_, length(coef(glm_mod))), names(coef(glm_mod)))
  out[names(cr_se)] <- cr_se
  out
}

# --- Helper: write one stargazer table for a (muster, primary, seats) triplet
write_conley_table <- function(mods, cutoff_km, label_suffix, title, out_path,
                               cov_labels) {
  ses <- list(
    conley_se(mods$muster,  cutoff_km, "logit"),
    conley_se(mods$primary, cutoff_km, "logit"),
    conley_se(mods$seat,    cutoff_km, "poisson")
  )
  stargazer(
    mods$muster, mods$primary, mods$seat,
    type             = "latex",
    se               = ses,
    title            = title,
    label            = paste0("tab:conley", label_suffix, "_", cutoff_km, "km"),
    omit             = hide_vars,
    covariate.labels = cov_labels,
    column.labels    = c("Muster", "Primary", "Seats"),
    add.lines        = list(
      c("Conley cutoff (km)", rep(as.character(cutoff_km), 3)),
      c("Kernel",             rep("Bartlett", 3))
    ),
    align            = TRUE,
    column.sep.width = ".5pt",
    omit.stat        = c("aic"),
    table.placement  = "H",
    out              = out_path
  )
}

# --- Loop all 9 monastic specs -------------------------------------------
for (spec_name in names(monastic_specs)) {
  spec      <- monastic_specs[[spec_name]]
  mods      <- full_models_by_spec[[spec_name]]
  mon_vars  <- spec$monastic_vars
  sfx       <- spec$suffix

  cov_order  <- c(mon_vars, "mg_fsnub", "mg_court",
                  "lLStax_pc", "wet_1535", "wet_1536", "lpopC", "distScot")
  cov_labels <- unlist(pretty_dict[cov_order])

  for (k in conley_cutoffs) {
    write_conley_table(
      mods         = mods,
      cutoff_km    = k,
      label_suffix = sfx,
      title        = paste0("Full Monastic Model — Conley SEs (", spec_name, ", ", k, " km)"),
      out_path     = paste0("Output/Tables/conley", sfx, "_", k, "km.tex"),
      cov_labels   = cov_labels
    )
  }
  cat("Wrote Conley tables for spec:", spec_name, "\n")
}

cat("\nAll Conley SE tables written to Output/Tables/conley_*.tex\n")


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:11
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Full Monastic Model — Conley SEs (total_raw, 50 km)} 
  \label{tab:conley_total_raw_50km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{muster \textasciitilde} & \multicolumn{1}{c}{primary \textasciitilde} & \multicolumn{1}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{Poisson}} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1.8ex] 
 ln

Warning message:
"st_centroid assumes attributes are constant over geometries"


...

Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"


Warning message:
"st_centroid assumes attributes are constant over geometries"



% Table created by stargazer v.5.2.3 by Marek Hlavac, Social Policy Institute. E-mail: marek.hlavac at gmail.com
% Date and time: Wed, May 13, 2026 - 14:10:42
% Requires LaTeX packages: dcolumn 
\begin{table}[H] \centering 
  \caption{Full Monastic Model — Conley SEs (ownOther_arak, 150 km)} 
  \label{tab:conley_ownOther_arak_150km} 
\begin{tabular}{@{\extracolsep{.5pt}}lD{.}{.}{-3} D{.}{.}{-3} D{.}{.}{-3} } 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{1}{c}{muster \textasciitilde} & \multicolumn{1}{c}{primary \textasciitilde} & \multicolumn{1}{c}{seats \textasciitilde} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{logistic}} & \multicolumn{1}{c}{\textit{Poisson}} \\ 
 & \multicolumn{1}{c}{Muster} & \multicolumn{1}{c}{Primary} & \multicolumn{1}{c}{Seats} \\ 
\\[-1.8ex] & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)}\\ 
\hline \\[-1


All Conley SE tables written to Output/Tables/conley_*.tex


## Section 4: Moran's I — Spatial Autocorrelation Diagnostics

Tests for residual spatial dependence. Reports Moran's I on:
- raw outcomes (`muster`, `primary`, `seats`),
- Pearson residuals from the full monastic model (`total` spec),
- Pearson residuals from the joint elite model.

Two spatial-weights matrices: row-standardized **queen contiguity** (primary)
and **k-nearest neighbors** with k=8 on parish centroids (robustness).

In [6]:
# Moran's I on raw outcomes and GLM Pearson residuals.
pacman::p_load(spdep, xtable)

# --- Build spatial weights once on the full parish set -------------------
nb_queen <- spdep::poly2nb(pdf, queen = TRUE)
lw_queen <- spdep::nb2listw(nb_queen, style = "W", zero.policy = TRUE)

parish_xy_nb01 <- sf::st_coordinates(sf::st_centroid(sf::st_geometry(pdf)))
nb_knn8  <- spdep::knn2nb(spdep::knearneigh(parish_xy_nb01, k = 8))
lw_knn8  <- spdep::nb2listw(nb_knn8, style = "W")

# --- Helper: Moran's I for a numeric vector aligned to pdf rows ----------
morans_row <- function(label, x, keep_rows, lw, lw_label) {
  if (length(keep_rows) < length(lw$neighbours)) {
    lw_use <- spdep::subset.listw(lw, subset = seq_along(lw$neighbours) %in% keep_rows,
                                  zero.policy = TRUE)
  } else {
    lw_use <- lw
  }
  mt <- spdep::moran.test(x, lw_use, zero.policy = TRUE, na.action = na.omit)
  data.frame(
    Model    = label,
    Weights  = lw_label,
    n        = length(x),
    Morans_I = unname(mt$estimate["Moran I statistic"]),
    Expected = unname(mt$estimate["Expectation"]),
    Variance = unname(mt$estimate["Variance"]),
    z        = unname(mt$statistic),
    p_value  = unname(mt$p.value)
  )
}

# --- Raw outcomes --------------------------------------------------------
raw_inputs <- list(
  muster  = pdf$muster,
  primary = pdf$primary,
  seats   = pdf$seats
)

# --- Residuals: full monastic models (split_arak primary spec) -----------
# Use split_arak as the representative per-arable-km² specification.
mon_mods_ref <- full_models_by_spec[["split_arak"]]

resid_inputs <- list(
  "Monastic — Muster"  = mon_mods_ref$muster,
  "Monastic — Primary" = mon_mods_ref$primary,
  "Monastic — Seats"   = mon_mods_ref$seat
)

# --- Build the table ------------------------------------------------------
rows <- list()
for (lw_lab in c("Queen", "KNN-8")) {
  lw_i <- if (lw_lab == "Queen") lw_queen else lw_knn8

  for (nm in names(raw_inputs)) {
    x    <- raw_inputs[[nm]]
    keep <- which(!is.na(x))
    rows[[length(rows) + 1L]] <- morans_row(
      paste0("Raw — ", nm), x[keep], keep, lw_i, lw_lab
    )
  }
  for (nm in names(resid_inputs)) {
    mod  <- resid_inputs[[nm]]
    used <- as.integer(rownames(model.frame(mod)))
    r    <- residuals(mod, type = "pearson")
    rows[[length(rows) + 1L]] <- morans_row(nm, r, used, lw_i, lw_lab)
  }
}
moran_tab_nb01 <- do.call(rbind, rows)
print(moran_tab_nb01, row.names = FALSE)

# --- Write LaTeX ---------------------------------------------------------
moran_tab_fmt <- moran_tab_nb01
moran_tab_fmt$Morans_I <- sprintf("%.3f", moran_tab_fmt$Morans_I)
moran_tab_fmt$Expected <- sprintf("%.4f", moran_tab_fmt$Expected)
moran_tab_fmt$Variance <- sprintf("%.5f", moran_tab_fmt$Variance)
moran_tab_fmt$z        <- sprintf("%.2f", moran_tab_fmt$z)
moran_tab_fmt$p_value  <- ifelse(moran_tab_fmt$p_value < 1e-4, "<1e-4",
                                 sprintf("%.4f", moran_tab_fmt$p_value))
colnames(moran_tab_fmt) <- c("Model", "Weights", "n", "Moran's I",
                             "E[I]", "Var(I)", "z", "p")

print(
  xtable(moran_tab_fmt,
         caption = "Moran's I — raw outcomes and Pearson residuals (split_arak spec).",
         label   = "tab:moran_nb01",
         align   = c("l", "l", "l", "r", "r", "r", "r", "r", "r")),
  include.rownames  = FALSE,
  caption.placement = "top",
  table.placement   = "H",
  file = "Output/Tables/morans_nb01.tex"
)
cat("Wrote Output/Tables/morans_nb01.tex\n")


Warning message in spdep::poly2nb(pdf, queen = TRUE):
"some observations have no neighbours;
if this seems unexpected, try increasing the snap argument."


Warning message in spdep::poly2nb(pdf, queen = TRUE):
"neighbour object has 5 sub-graphs;
if this sub-graph count seems unexpected, try increasing the snap argument."


Warning message in subset.nb(x = nb, subset = subset):
"subsetting caused increase in subgraph count"


Warning message in subset.nb(x = nb, subset = subset):
"subsetting caused increase in subgraph count"


Warning message in subset.nb(x = nb, subset = subset):
"subsetting caused increase in subgraph count"


              Model Weights    n    Morans_I      Expected     Variance
       Raw — muster   Queen 1755  0.03985870 -0.0005714286 0.0002106817
      Raw — primary   Queen 1755  0.02411574 -0.0005714286 0.0002067796
        Raw — seats   Queen 1755  0.16299498 -0.0005714286 0.0002087765
  Monastic — Muster   Queen 1391  0.01694707 -0.0007215007 0.0002644560
 Monastic — Primary   Queen 1391 -0.01087221 -0.0007215007 0.0002420891
   Monastic — Seats   Queen 1391  0.06011523 -0.0007215007 0.0002714619
       Raw — muster   KNN-8 1755  0.04804414 -0.0005701254 0.0001272905
      Raw — primary   KNN-8 1755  0.04435592 -0.0005701254 0.0001249379
        Raw — seats   KNN-8 1755  0.13811735 -0.0005701254 0.0001261418
  Monastic — Muster   KNN-8 1391  0.02697648 -0.0007215007 0.0001821314
 Monastic — Primary   KNN-8 1391  0.02878659 -0.0007215007 0.0001667232
   Monastic — Seats   KNN-8 1391  0.09371822 -0.0007215007 0.0001869576
         z      p_value
  2.785427 2.672866e-03
  1.716791 4.300

Wrote Output/Tables/morans_nb01.tex
